In [2]:
# IMPORT LIBRARIES
import sys
import subprocess
import warnings
warnings.filterwarnings('ignore')

def ensure_package(pkg_name, import_name=None):
    """Install package if import fails."""
    import importlib
    try:
        importlib.import_module(import_name or pkg_name)
    except Exception:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg_name])
        except Exception as e:
            print(f'Package install skipped/failed for {pkg_name}: {e}')

for pkg, imp in [
    ('pandas', 'pandas'),
    ('numpy', 'numpy'),
    ('matplotlib', 'matplotlib'),
    ('scikit-learn', 'sklearn'),
    ('seaborn', 'seaborn'),
    ('xgboost', 'xgboost'),
    ('shap', 'shap'),
]:
    ensure_package(pkg, imp)

import os
import re
import json
import math
import textwrap
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    confusion_matrix, classification_report
)
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.base import clone
from xgboost import XGBClassifier
import shap

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
sns.set_theme(style='whitegrid')
np.random.seed(42)
random.seed(42)

df = pd.read_csv('/content/Foodpanda Analysis Dataset.csv')

def infer_units_and_allowed_values(df, col):
    dtype = df[col].dtype
    unit = 'N/A'
    allowed_values = 'N/A'

    if pd.api.types.is_numeric_dtype(dtype):
        unit = 'Numeric'
        if col == 'price':
            unit = 'Currency (e.g., USD)'
        elif col == 'quantity':
            unit = 'Units'
        elif col == 'order_frequency':
            unit = 'Days'
        elif col == 'loyalty_points':
            unit = 'Points'
        elif col == 'rating':
            unit = 'Scale (1-5)'
        allowed_values = f"Min: {df[col].min()}, Max: {df[col].max()}"
    elif pd.api.types.is_datetime64_any_dtype(dtype) or 'date' in col.lower():
        unit = 'Date'
        # For date columns, display min and max dates
        min_date = pd.to_datetime(df[col], errors='coerce').min()
        max_date = pd.to_datetime(df[col], errors='coerce').max()
        allowed_values = f"Min Date: {min_date}, Max Date: {max_date}"
    elif pd.api.types.is_object_dtype(dtype):
        unique_vals = df[col].dropna().unique()
        if len(unique_vals) <= 20: # If few unique values, list them
            unit = 'Categorical'
            allowed_values = ', '.join(map(str, unique_vals))
        else: # If many unique values, describe them generally
            unit = 'Text/High Cardinality'
            # Limit to first 5 examples if many unique values
            allowed_values = f"Too many unique values ({len(unique_vals)}) to list. Examples: {', '.join(map(str, unique_vals[:5]))}..."
            if col in ['customer_id', 'order_id']:
                unit = 'Identifier'

    return unit, allowed_values

print('STEP 2: DATA DICTIONARY')

data_dict_rows = []
for col in df.columns:
    unit, allowed = infer_units_and_allowed_values(df, col)
    data_dict_rows.append({
        'variable': col,
        'type': str(df[col].dtype),
        'unit': unit,
        'allowed_values': allowed
    })
data_dictionary = pd.DataFrame(data_dict_rows)
display(data_dictionary)

STEP 2: DATA DICTIONARY


,variable,type,unit,allowed_values
0,customer_id,object,Identifier,Too many unique values (6000) to list. Example...
1,gender,object,Categorical,"Male, Other, Female"
2,age,object,Categorical,"Senior, Adult, Teenager"
3,city,object,Categorical,"Lahore, Multan, Peshawar, Islamabad, Karachi"
4,signup_date,object,Date,"Min Date: 2023-08-22 00:00:00, Max Date: 2025-..."
5,order_id,object,Identifier,Too many unique values (6000) to list. Example...
6,order_date,object,Date,"Min Date: 2023-08-23 00:00:00, Max Date: 2025-..."
7,restaurant_name,object,Categorical,"McDonald's, KFC, Pizza Hut, Subway, Burger King"
8,dish_name,object,Categorical,"Burger, Fries, Pizza, Sandwich, Pasta"
9,category,object,Categorical,"Italian, Dessert, Chinese, Fast Food, Continental"
